# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

### Paso 1: Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# 1. Ruta del archivo de texto
ruta_turismo = "..\\data\\01_corpus_turismo_500.txt"

# 2. Leer el contenido del archivo
# Cada línea del archivo se tratará como un documento individual
with open(ruta_turismo, "r", encoding="utf-8") as archivo:
    lineas_turismo = archivo.readlines()

# 3. Configurar el modelo TF-IDF
# Usamos stop_words en español para ignorar palabras como "el", "la", "de", etc.
vectorizador_turismo = TfidfVectorizer(stop_words=['de', 'la', 'en', 'el', 'y', 'un', 'una', 'es', 'por'])

# 4. Calcular la matriz TF-IDF (Frecuencia de término - Frecuencia inversa de documento)
matriz_tfidf_turismo = vectorizador_turismo.fit_transform(lineas_turismo)

print(f"Matriz de turismo generada con {len(lineas_turismo)} documentos.")
print(f"Número de palabras únicas (columnas): {matriz_tfidf_turismo.shape[1]}")

Matriz de turismo generada con 500 documentos.
Número de palabras únicas (columnas): 108


In [2]:
# Obtenemos los nombres de las palabras
palabras_turismo = vectorizador_turismo.get_feature_names_out()

# Sumamos los puntajes TF-IDF de cada palabra en todos los documentos
pesos_totales = matriz_tfidf_turismo.sum(axis=0).A1

# Creamos un ranking para ver qué palabras definen mejor tu corpus
ranking_turismo = pd.DataFrame({
    'Palabra': palabras_turismo,
    'Peso_TFIDF': pesos_totales
}).sort_values(by='Peso_TFIDF', ascending=False)

print("Top 10 palabras más importantes del corpus de turismo:")
print(ranking_turismo.head(10))

Top 10 palabras más importantes del corpus de turismo:
         Palabra  Peso_TFIDF
74          para   48.035823
92            su   41.385495
48         ideal   34.829083
40       feriado   32.918689
81       próximo   28.508127
37   experiencia   25.746642
51   inolvidable   25.746642
78      perfecto   24.809593
62         lugar   22.717088
105      visitar   22.717088


### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

In [32]:
import os

# 1. Definir la carpeta donde están los libros .txt
carpeta_libros = "..\\data\\1000Libros"
lista_textos = []
nombres_de_libros = []

# 2. Obtener la lista de archivos en la carpeta
archivos_en_carpeta = os.listdir(carpeta_libros)

# 3. Leer hasta 1000 archivos y guardarlos en una lista
for nombre_archivo in archivos_en_carpeta[:1000]:
    ruta_libro = os.path.join(carpeta_libros, nombre_archivo)
    
    # leer el archivo 
    try:
        with open(ruta_libro, "r", encoding="utf-8") as f:
            contenido = f.read()
            lista_textos.append(contenido)
            nombres_de_libros.append(nombre_archivo)
    except Exception as e:
        print(f"No se pudo leer el archivo {nombre_archivo}: {e}")

print(f"Se han cargado {len(lista_textos)} libros al corpus Gutenberg.")

Se han cargado 1000 libros al corpus Gutenberg.


### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

In [33]:
import time

# Capturamos el tiempo de inicio
inicio_tfidf = time.time()

# 1. Configurar el vectorizador para los libros
# max_features=5000 ayuda a que la matriz no sea gigante, tomando solo las palabras más importantes
vectorizador_gutenberg = TfidfVectorizer(stop_words=['de', 'la', 'en', 'el', 'y', 'un', 'una', 'es', 'por'], 
                                         max_features=5000)

# 2. Transformar los libros en la matriz TF-IDF
matriz_tfidf_gutenberg = vectorizador_gutenberg.fit_transform(lista_textos)

# Capturamos el tiempo de fin
fin_tfidf = time.time()

print("Cálculo de TF-IDF completado para el corpus Gutenberg.")
print(f"Dimensiones de la matriz (Libros, Palabras): {matriz_tfidf_gutenberg.shape}")
print(f"Tiempo total de ejecución: {fin_tfidf - inicio_tfidf:.4f} segundos")

Cálculo de TF-IDF completado para el corpus Gutenberg.
Dimensiones de la matriz (Libros, Palabras): (1000, 5000)
Tiempo total de ejecución: 70.0832 segundos


In [34]:
# --- 3. Verificación de ejecución y análisis de datos ---

# Extraemos los nombres de las 5000 palabras (columnas)
nombres_palabras = vectorizador_gutenberg.get_feature_names_out()

# Calculamos la importancia total de cada palabra en todo el corpus
importancia_total = matriz_tfidf_gutenberg.sum(axis=0).A1

# 1. Creamos un Ranking de las palabras con más peso
ranking_palabras = pd.DataFrame({
    'Palabra': nombres_palabras,
    'Peso_Total_TFIDF': importancia_total
}).sort_values(by='Peso_Total_TFIDF', ascending=False)

print(f"Cálculo completado. Matriz de: {matriz_tfidf_gutenberg.shape[0]} libros x {matriz_tfidf_gutenberg.shape[1]} palabras.")
print("\n--- Top 10 palabras con mayor relevancia en el corpus ---")
print(ranking_palabras.head(10))

# 2. Mostramos una "rebanada" de la matriz (primeros 5 libros y 10 palabras)
df_vista_previa = pd.DataFrame(
    matriz_tfidf_gutenberg[:5, :10].toarray(), 
    columns=nombres_palabras[:10]
)

print("\n--- Vista previa de valores TF-IDF (Primeros 5 libros) ---")
print(df_vista_previa)

Cálculo completado. Matriz de: 1000 libros x 5000 palabras.

--- Top 10 palabras con mayor relevancia en el corpus ---
     Palabra  Peso_Total_TFIDF
3792     que        558.582644
2699     los        273.970037
4135      se        210.347031
3110      no        200.236447
2545     las        180.952195
892      con        177.198064
1246     del        160.722045
4347      su        159.931521
188       al        116.480626
2686      lo        104.299503

--- Vista previa de valores TF-IDF (Primeros 5 libros) ---
        000       10       100        11        12        13        14  \
0  0.000000  0.00000  0.000000  0.000000  0.000000  0.000000  0.000000   
1  0.000000  0.00000  0.000000  0.000000  0.000000  0.000000  0.000000   
2  0.022474  0.02198  0.001409  0.018029  0.017133  0.015173  0.013381   
3  0.022474  0.02198  0.001409  0.018029  0.017133  0.015173  0.013381   
4  0.004565  0.00000  0.000366  0.000000  0.004245  0.000000  0.000000   

         15        16        17  
0

### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

In [30]:
from sklearn.metrics.pairwise import cosine_similarity

def buscar_documento(consulta, vectorizador, matriz_tfidf, titulos):
    """
    Busca la consulta dentro de la matriz TF-IDF y devuelve los resultados más parecidos.
    """
    # 1. Convertir la frase de búsqueda (consulta) al mismo formato vectorial que los libros
    vector_consulta = vectorizador.transform([consulta])
    
    # 2. Comparar el vector de la consulta contra todos los documentos (Similitud de Coseno)
    similitudes = cosine_similarity(vector_consulta, matriz_tfidf).flatten()
    
    # 3. Ordenar los resultados de mayor a menor similitud
    indices_mas_cercanos = similitudes.argsort()[::-1]
    
    print(f"\nResultados para la búsqueda: '{consulta}'")
    print("-" * 50)
    
    encontrado = False
    # Mostramos los 5 resultados con mayor puntaje
    for i in indices_mas_cercanos[:5]:
        if similitudes[i] > 0:
            print(f"Documento: {titulos[i]} | Similitud: {similitudes[i]:.4f}")
            encontrado = True
            
    if not encontrado:
        print("No se encontraron coincidencias relevantes.")

In [31]:
# --- EJECUCIÓN ---
# Probamos la búsqueda usando las variables del corpus de 1000 libros
# Cambia el texto entre comillas por lo que quieras buscar
buscar_documento("aventuras en la selva y expediciones", 
                 vectorizador_gutenberg, 
                 matriz_tfidf_gutenberg, 
                 nombres_de_libros)


Resultados para la búsqueda: 'aventuras en la selva y expediciones'
--------------------------------------------------
Documento: El libro de las tierras vírgenes.txt | Similitud: 0.0811
Documento: Azul... Obras Completas Vol. IV.txt | Similitud: 0.0201
Documento: Los pescadores de Trépang.txt | Similitud: 0.0187
Documento: Cantos de Vida y Esperanza, Los Cisnes y otros poemas. Obras Completas Vol. VII.txt | Similitud: 0.0183
Documento: Cañas y barro Novela.txt | Similitud: 0.0164
